In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_columns', None)

In [2]:
#importing the filtered powerlifting data
#source: OpenPowerlifting (openpowerlifting.org) - pre-filtered to Raw SBD meets 2015+, lifters with 2+ meets
df = pd.read_csv('data/raw/openpowerlifting_filtered.csv')

print(df.shape)
df.head()

(714698, 10)


,Name,Sex,Age,BodyweightKg,Best3SquatKg,Best3BenchKg,Best3DeadliftKg,TotalKg,Country,Date
0,A I Temaki,M,35.5,154.70,370.0,212.0,235.0,817.0,Nauru,2024-04-13
1,A I Temaki,M,36.0,145.45,400.0,220.0,235.0,855.0,Nauru,2025-03-29
2,A Madhuri,F,14.5,74.50,112.5,50.0,127.5,290.0,NaN,2022-04-09
3,A Madhuri,F,15.5,75.27,117.5,55.0,115.0,287.5,NaN,2023-06-08
4,A Madhuri,F,16.5,76.00,115.0,60.0,125.0,300.0,NaN,2024-06-06


In [3]:
#check what columns we have and their types
print(df.dtypes)
print('---')
print(df.isna().sum())

Name                object
Sex                 object
Age                float64
BodyweightKg       float64
Best3SquatKg       float64
Best3BenchKg       float64
Best3DeadliftKg    float64
TotalKg            float64
Country             object
Date                object
dtype: object
---
Name                    0
Sex                     0
Age                     0
BodyweightKg            0
Best3SquatKg          102
Best3BenchKg          125
Best3DeadliftKg       109
TotalKg                 0
Country            133579
Date                    0
dtype: int64


In [4]:
#convert Date to datetime so we can sort meets in order per lifter
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

#sort by lifter name then date - this is important because we need meets in chronological order
#to calculate whether the lifter improved from their last meet
df = df.sort_values(['Name', 'Date']).reset_index(drop=True)

In [5]:
# ---- CREATE LAG FEATURES ----
# For each lifter, we want to know what they lifted at their PREVIOUS meet
# This is the same idea as looking at a user's previous workout in Milo
# groupby Name then shift(1) gives us the row from one meet ago

df['prev_total']     = df.groupby('Name')['TotalKg'].shift(1)
df['prev_squat']     = df.groupby('Name')['Best3SquatKg'].shift(1)
df['prev_bench']     = df.groupby('Name')['Best3BenchKg'].shift(1)
df['prev_deadlift']  = df.groupby('Name')['Best3DeadliftKg'].shift(1)
df['prev_bodyweight']= df.groupby('Name')['BodyweightKg'].shift(1)
df['prev_date']      = df.groupby('Name')['Date'].shift(1)

#rows where prev values are NaN are the first meet of a lifter - we drop these
#because there is nothing to compare against
df = df.dropna(subset=['prev_total', 'prev_squat', 'prev_bench', 'prev_deadlift']).reset_index(drop=True)

print('Shape after removing first-meet rows:', df.shape)

Shape after removing first-meet rows: (544711, 16)


In [6]:
# ---- CREATE TARGET VARIABLE ----
# improved = 1 if the lifter increased their total kg compared to their previous meet
# improved = 0 if they stayed the same or went down
# This maps directly to 'should_increase_weight' in the Milo app context

df['improved'] = (df['TotalKg'] > df['prev_total']).astype(int)

print(df['improved'].value_counts())
print('---')
print(round(df['improved'].value_counts(normalize=True), 3))

improved
1    354242
0    190469
Name: count, dtype: int64
---
improved
1    0.65
0    0.35
Name: proportion, dtype: float64


In [7]:
# ---- DAYS BETWEEN MEETS ----
# How long did the lifter rest between competitions?
# Shorter breaks may mean less recovery - similar to tracking rest days in Milo

df['days_between_meets'] = (df['Date'] - df['prev_date']).dt.days

# check the distribution before binning
print(df['days_between_meets'].describe())
print(df['days_between_meets'].value_counts().sort_index().head(20))

count    544711.000000
mean        244.535355
std         314.573998
min           0.000000
25%          78.000000
50%         162.000000
75%         294.000000
max        4047.000000
Name: days_between_meets, dtype: float64
days_between_meets
0     61207
1       411
2        59
3        86
4        68
5       130
6       230
7       825
8       287
9       149
10      132
11      320
12      210
13      453
14     1103
15      391
16      181
17      377
18      256
19      274
Name: count, dtype: int64


In [8]:
# bin days between meets into bands - similar to how we did timespan bands in the Dior project
# <30 days between meets is very quick turnaround
# 30-90 days is a typical competition cycle
# 90-180 days is a longer prep
# 180+ days is a long break

bins_days = [0, 30, 90, 180, 365, np.inf]
labels_days = ['under_30', '30_to_90', '90_to_180', '180_to_365', 'over_365']

df['days_band'] = pd.cut(df['days_between_meets'], bins=bins_days, labels=labels_days, right=False)

print(df['days_band'].value_counts())

days_band
180_to_365    156871
90_to_180     145694
over_365       92846
under_30       76458
30_to_90       72842
Name: count, dtype: int64


In [9]:
# ---- BODYWEIGHT CHANGE ----
# Did the lifter gain or lose weight between meets?
# big weight cuts can hurt performance, weight gain usually signals muscle gain

df['bodyweight_change'] = df['BodyweightKg'] - df['prev_bodyweight']

# bin into bands
bins_bw = [-np.inf, -5, -2, 2, 5, np.inf]
labels_bw = ['lost_5plus', 'lost_2_to_5', 'stable', 'gained_2_to_5', 'gained_5plus']

df['bodyweight_change_band'] = pd.cut(df['bodyweight_change'], bins=bins_bw, labels=labels_bw, right=False)

print(df['bodyweight_change_band'].value_counts())

bodyweight_change_band
stable           355267
gained_2_to_5     66784
gained_5plus      60135
lost_2_to_5       34898
lost_5plus        27627
Name: count, dtype: int64


In [10]:
# ---- PREVIOUS TOTAL BAND ----
# How strong was the lifter at their previous meet?
# We use the previous total (not current) to avoid data leakage into our target
# These bands are roughly beginner / intermediate / advanced / elite

# check distribution to decide bands
print(df['prev_total'].describe())

count    544711.000000
mean        474.835162
std         157.351150
min           7.000000
25%         342.500000
50%         477.500000
75%         592.500000
max        1153.500000
Name: prev_total, dtype: float64


In [11]:
bins_total = [0, 300, 450, 600, 750, np.inf]
labels_total = ['under_300', '300_to_450', '450_to_600', '600_to_750', 'over_750']

df['prev_total_band'] = pd.cut(df['prev_total'], bins=bins_total, labels=labels_total, right=False)

#do the same for individual lifts because each lift may have its own relationship with improvement
bins_squat = [0, 100, 150, 200, 250, np.inf]
labels_squat = ['sq_under_100', 'sq_100_to_150', 'sq_150_to_200', 'sq_200_to_250', 'sq_over_250']
df['prev_squat_band'] = pd.cut(df['prev_squat'], bins=bins_squat, labels=labels_squat, right=False)

bins_bench = [0, 60, 90, 120, 150, np.inf]
labels_bench = ['bp_under_60', 'bp_60_to_90', 'bp_90_to_120', 'bp_120_to_150', 'bp_over_150']
df['prev_bench_band'] = pd.cut(df['prev_bench'], bins=bins_bench, labels=labels_bench, right=False)

bins_dl = [0, 120, 175, 230, 280, np.inf]
labels_dl = ['dl_under_120', 'dl_120_to_175', 'dl_175_to_230', 'dl_230_to_280', 'dl_over_280']
df['prev_deadlift_band'] = pd.cut(df['prev_deadlift'], bins=bins_dl, labels=labels_dl, right=False)

print(df['prev_total_band'].value_counts())

prev_total_band
450_to_600    171718
300_to_450    160287
600_to_750    109000
under_300      82701
over_750       21005
Name: count, dtype: int64


In [12]:
# ---- AGE BAND ----
# Younger lifters tend to improve faster, older lifters plateau
# We use age bands instead of raw age to reduce noise and make it a dummy-able feature

print(df['Age'].describe())

count    544711.000000
mean         29.459367
std          12.169966
min           5.000000
25%          21.000000
50%          25.500000
75%          35.000000
max          93.500000
Name: Age, dtype: float64


In [13]:
bins_age = [0, 20, 25, 30, 35, 45, np.inf]
labels_age = ['under_20', '20_to_25', '25_to_30', '30_to_35', '35_to_45', 'over_45']

df['age_band'] = pd.cut(df['Age'], bins=bins_age, labels=labels_age, right=False)

print(df['age_band'].value_counts())

age_band
20_to_25    148930
under_20    104396
25_to_30     92738
35_to_45     73541
over_45      66910
30_to_35     58196
Name: count, dtype: int64


In [14]:
# ---- HANDLE MISSING VALUES ----
# After all the feature engineering let's check what is still missing

print('Missing values per column:')
print(df.isna().sum().sort_values(ascending=False))

Missing values per column:
Country                   93566
Best3BenchKg                 57
Best3DeadliftKg              57
Best3SquatKg                 44
Name                          0
prev_date                     0
prev_deadlift_band            0
prev_bench_band               0
prev_squat_band               0
prev_total_band               0
bodyweight_change_band        0
bodyweight_change             0
days_band                     0
days_between_meets            0
improved                      0
prev_deadlift                 0
prev_bodyweight               0
Sex                           0
prev_bench                    0
prev_squat                    0
prev_total                    0
Date                          0
TotalKg                       0
BodyweightKg                  0
Age                           0
age_band                      0
dtype: int64


In [15]:
#drop rows with any remaining NaN values
#mostly from edge cases in the band cuts or missing age/bodyweight data
df = df.dropna().reset_index(drop=True)

print('Shape after dropna:', df.shape)

Shape after dropna: (451065, 26)


In [16]:
# ---- CREATE DUMMY VARIABLES ----
# Same approach as in the Dior project - turn all categorical bands into boolean columns
# We also drop one dummy per group to avoid the dummy variable trap

# Sex - already binary so just map it
df['is_female'] = (df['Sex'] == 'F').astype(bool)

# all the band columns get get_dummies
cols_to_dummy = ['days_band', 'bodyweight_change_band',
                 'prev_total_band', 'prev_squat_band',
                 'prev_bench_band', 'prev_deadlift_band', 'age_band']

for col in cols_to_dummy:
    dummies = pd.get_dummies(df[col], prefix=col, dtype=bool)
    df = pd.concat([df, dummies], axis=1)

df = df.drop(columns=cols_to_dummy)

print(df.shape)
df.head(3)

(451065, 56)


,Name,Sex,Age,BodyweightKg,Best3SquatKg,Best3BenchKg,Best3DeadliftKg,TotalKg,Country,Date,prev_total,prev_squat,prev_bench,prev_deadlift,prev_bodyweight,prev_date,improved,days_between_meets,bodyweight_change,is_female,days_band_under_30,days_band_30_to_90,days_band_90_to_180,days_band_180_to_365,days_band_over_365,bodyweight_change_band_lost_5plus,bodyweight_change_band_lost_2_to_5,bodyweight_change_band_stable,bodyweight_change_band_gained_2_to_5,bodyweight_change_band_gained_5plus,prev_total_band_under_300,prev_total_band_300_to_450,prev_total_band_450_to_600,prev_total_band_600_to_750,prev_total_band_over_750,prev_squat_band_sq_under_100,prev_squat_band_sq_100_to_150,prev_squat_band_sq_150_to_200,prev_squat_band_sq_200_to_250,prev_squat_band_sq_over_250,prev_bench_band_bp_under_60,prev_bench_band_bp_60_to_90,prev_bench_band_bp_90_to_120,prev_bench_band_bp_120_to_150,prev_bench_band_bp_over_150,prev_deadlift_band_dl_under_120,prev_deadlift_band_dl_120_to_175,prev_deadlift_band_dl_175_to_230,prev_deadlift_band_dl_230_to_280,prev_deadlift_band_dl_over_280,age_band_under_20,age_band_20_to_25,age_band_25_to_30,age_band_30_to_35,age_band_35_to_45,age_band_over_45
0,A I Temaki,M,36.0,145.45,400.0,220.0,235.0,855.0,Nauru,2025-03-29,817.0,370.0,212.0,235.0,154.7,2024-04-13,1,350,-9.25,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,True,False,False,False,True,False,False,False,False,False,True,False
1,A Phi Le,M,18.5,55.00,190.0,112.5,197.5,500.0,USA,2025-02-08,472.5,182.5,105.0,185.0,54.5,2024-11-09,1,91,0.50,False,False,False,True,False,False,False,False,True,False,False,False,False,True,False,False,False,False,True,False,False,False,False,True,False,False,False,False,True,False,False,True,False,False,False,False,False
2,A Phi Le,M,19.0,55.95,196.5,117.5,212.5,526.5,USA,2025-04-03,500.0,190.0,112.5,197.5,55.0,2025-02-08,1,54,0.95,False,False,True,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,True,False,False,False,False,True,False,False,False,False,True,False,False,True,False,False,False,False,False


In [17]:
# ---- SELECT FINAL COLUMNS ----
# We only keep what the model will actually use
# Raw numeric values are dropped in favour of the banded dummies
# to keep the same structure as the Dior project

# columns to keep for the model
keep_cols = ['improved', 'is_female'] + [c for c in df.columns if any(
    c.startswith(p) for p in ['days_band_', 'bodyweight_change_band_',
                               'prev_total_band_', 'prev_squat_band_',
                               'prev_bench_band_', 'prev_deadlift_band_', 'age_band_'])]

df_clean = df[keep_cols].copy()

print('Final shape:', df_clean.shape)
print('Columns:', df_clean.columns.tolist())
print('---')
print('Class balance:')
print(round(df_clean['improved'].value_counts(normalize=True), 3))

Final shape: (451065, 38)
Columns: ['improved', 'is_female', 'days_band_under_30', 'days_band_30_to_90', 'days_band_90_to_180', 'days_band_180_to_365', 'days_band_over_365', 'bodyweight_change_band_lost_5plus', 'bodyweight_change_band_lost_2_to_5', 'bodyweight_change_band_stable', 'bodyweight_change_band_gained_2_to_5', 'bodyweight_change_band_gained_5plus', 'prev_total_band_under_300', 'prev_total_band_300_to_450', 'prev_total_band_450_to_600', 'prev_total_band_600_to_750', 'prev_total_band_over_750', 'prev_squat_band_sq_under_100', 'prev_squat_band_sq_100_to_150', 'prev_squat_band_sq_150_to_200', 'prev_squat_band_sq_200_to_250', 'prev_squat_band_sq_over_250', 'prev_bench_band_bp_under_60', 'prev_bench_band_bp_60_to_90', 'prev_bench_band_bp_90_to_120', 'prev_bench_band_bp_120_to_150', 'prev_bench_band_bp_over_150', 'prev_deadlift_band_dl_under_120', 'prev_deadlift_band_dl_120_to_175', 'prev_deadlift_band_dl_175_to_230', 'prev_deadlift_band_dl_230_to_280', 'prev_deadlift_band_dl_over_2

In [18]:
# ---- CHECK CORRELATIONS ----
# Same check as in the Dior project - if anything has an unrealistic correlation
# it would indicate data leakage and we would need to remove it

corr = df_clean.corr(numeric_only=True)['improved'].sort_values(ascending=False)
print(corr)

improved                                1.000000
bodyweight_change_band_gained_5plus     0.176582
days_band_180_to_365                    0.154486
bodyweight_change_band_gained_2_to_5    0.148646
days_band_90_to_180                     0.122128
age_band_25_to_30                       0.084558
days_band_over_365                      0.072487
prev_deadlift_band_dl_under_120         0.057277
prev_total_band_under_300               0.056228
prev_squat_band_sq_under_100            0.047888
prev_bench_band_bp_under_60             0.047583
age_band_30_to_35                       0.046962
age_band_under_20                       0.046343
days_band_30_to_90                      0.045924
prev_bench_band_bp_90_to_120            0.032078
prev_total_band_450_to_600              0.029974
prev_deadlift_band_dl_175_to_230        0.028576
prev_squat_band_sq_150_to_200           0.028295
prev_squat_band_sq_100_to_150           0.014962
age_band_20_to_25                       0.010968
prev_deadlift_band_d

In [19]:
# ---- EXPORT CLEANED DATA ----
# this cleaned csv is what the models notebook will load

df_clean.to_csv('data/milo_clean.csv', index=False)
print('Saved to data/milo_clean.csv')
print('Shape:', df_clean.shape)

Saved to data/milo_clean.csv
Shape: (451065, 38)
